This notebook reads lidar and camera data from UR bag and converts it into a format that can be used by the evaluate_flow_calibration.py script

In [5]:
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image

from mcap_protobuf.decoder import DecoderFactory
from mcap.reader import make_reader

from bag_handler import read_all_messages_optimized, get_topic_info
from tkCloudProtoConverter import protoCloudToNumpy, protoCloudToPcdFile

In [19]:
DATA_FOLDER = Path("/data/ur")
LIDAR_TOPIC = "/lidar_fc/cloud"
CAMERA_TOPIC = "/camera_fl/image"

In [22]:
topic_message_counts = get_topic_info(str(DATA_FOLDER / "short_aligned.mcap"))
topic_message_counts = {topic: count for topic, type, count in topic_message_counts}
lidar_and_camera_frames = read_all_messages_optimized(
    bag_file=str(DATA_FOLDER / "short.mcap"),
    topics_to_read={LIDAR_TOPIC: 0, CAMERA_TOPIC: 1},
    topic_message_counts=topic_message_counts,
    frame_samples=9,
)

In [23]:
images = lidar_and_camera_frames['frame_samples'][CAMERA_TOPIC]
pointclouds = lidar_and_camera_frames['frame_samples'][LIDAR_TOPIC]
for i, (img_sample, pcd_sample) in enumerate(zip(images, pointclouds)):
    print(i, end=" ")
    nparr = np.frombuffer(img_sample["data"].data, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img, 'RGB')
    img.save((DATA_FOLDER / f"camera/{i:04d}.png").__str__())
    
    pcd_bin = protoCloudToPcdFile(pcd_sample["data"])
    with open(DATA_FOLDER / f"lidar/{i:04d}.pcd", "wb") as f:
        f.write(pcd_bin)
    

0 1 2 3 4 5 6 7 8 

### Old extraction code

In [19]:
pcd_sample = lidar_and_camera_frames['frame_samples'][LIDAR_TOPIC][0]["data"]
binary = protoCloudToPcdFile(pcd_sample)
bin_data = binary[binary.find(b"binary")+7:]

In [ ]:
data = lidar_and_camera_frames['frame_samples'][CAMERA_TOPIC][0]['data'].data
nparr = np.frombuffer(data, np.uint8)
img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
import matplotlib.pyplot as plt
plt.imshow(img)

In [ ]:
f = open(DATA_FOLDER / "short.mcap", "rb")
reader = make_reader(f, decoder_factories=[DecoderFactory()])

last_pcd = None
last_img = None
for schema, channel, message, proto_msg in tqdm(
        reader.iter_decoded_messages(topics=[LIDAR_TOPIC, CAMERA_TOPIC])
    ):
    if channel.topic == CAMERA_TOPIC:
        nparr = np.frombuffer(proto_msg.data, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        last_img = img
    elif channel.topic == LIDAR_TOPIC:
        last_pcd = protoCloudToPcdFile(proto_msg)

785it [00:06, 120.26it/s]


In [40]:
# save image and pointcloud
if last_img is not None:
    img = Image.fromarray(last_img, 'RGB')
    img.save((DATA_FOLDER / f"camera/{SERIAL}.png").__str__())
if last_pcd is not None:
    with open(DATA_FOLDER / f"lidar/{SERIAL}.pcd", "wb") as f:
        f.write(last_pcd)

# Test load PCD

Reading PCD file and comparing it to the original format to ensure that the data is being saved correctly.

In [6]:
with open(DATA_FOLDER / "lidar/0000.pcd", "rb") as f:
    pcd = f.read()

bin_data = pcd[pcd.find(b"binary\n")+7:]


In [15]:
bin_points = np.frombuffer(bin_data, dtype=np.float32)
bin_points.reshape(-1, 3).shape

(38261, 3)

In [4]:
points = protoCloudToNumpy(proto_msg)

In [7]:
points.shape

(38053, 6)